# EVAL 2 — tuned Qwen3-4B on held-out `eval.jsonl` (bare deployment prompt)

The base-vs-tuned **headline**. Each of the held-out `eval.jsonl` specs is fed to the tuned model as the **bare** system+user prompt it was trained on (no contract in the prompt — the contract lives in the weights). Every emitted `generate_crossword` program is scored through the **same** sandbox + scorer + real-dictionary check as Claude's EVAL 2 (`pipeline.eval_opus_fleet.score_one`).

**Comparison:** unaugmented Claude Opus scored **0/100** on these identical bare prompts (GAP_ANALYSIS EVAL 2).

> **No separate "as-is" run.** Claude's EVAL 2 also ran each program on its own interface to rule out an API-mismatch artifact — needed only because Claude didn't conform to our contract. The tuned model emits a conforming `generate_crossword` *function*, so calling it and scoring the layout (below) already is the own-terms test. (Cell 8 optionally dumps the raw programs for manual inspection.)

**Runtime:** GPU. L4 (24 GB) / A100 (40 GB) recommended; T4 works but generation is slower. **Order:** run top to bottom.

## 1. Get the code
Clone the repo (set your URL) or upload the folder and set `PROJECT_DIR`. The clone already contains everything the eval needs: `data/sft/eval.jsonl`, the word lists in `data/wordlists/` (incl. `words_alpha.txt`), and the `pipeline/` + `harness/` scoring code.

> Push your latest local changes first — the eval reflects whatever is committed here.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM.git"
import os
!git clone -q $REPO_URL slm || echo "clone skipped/failed — upload the folder instead"
PROJECT_DIR = "/content/slm"   # adjust if you uploaded elsewhere
assert os.path.isdir(os.path.join(PROJECT_DIR, "pipeline")), "Set PROJECT_DIR to the repo root"
%cd $PROJECT_DIR

## 2. GPU + install deps
Colab already ships torch. We add `transformers`/`accelerate` (pinned to the training snapshot) to load the merged model, and `wordfreq` (the English palette intersects the wordfreq top-N). No bitsandbytes — the merged model loads in 16-bit directly.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU (L4/A100 recommended)"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q "transformers==4.53.*" "accelerate==1.8.*" wordfreq

> **Expected pip warning — safe to ignore.** Colab's pre-installed `gradio` wants a newer `huggingface-hub` than `transformers 4.53` pins. `gradio` is unused here; do **not** upgrade `huggingface-hub` (it would break `transformers`).

## 3. Point to the merged tuned model (on Drive)
Mount Drive and set `MODEL_DIR` to your merged model folder (`…/qwen3-4b-crossword-qlora-merged`) — the full standalone model, not the adapter.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# EDIT this to your merged-model path on Drive:
MODEL_DIR = "/content/drive/MyDrive/qwen3-4b-crossword-qlora-merged"
import os
assert os.path.isdir(MODEL_DIR), f"MODEL_DIR not found: {MODEL_DIR}"
assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), \
    "not a full model dir (need config.json + model-*.safetensors, i.e. the -merged folder, not the adapter)"
print("model dir OK:", MODEL_DIR)

## 4. Load the tuned model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # left-pad so batched generation aligns at the prompt end
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
model.eval()
print("loaded:", model.config.model_type, "| dtype", next(model.parameters()).dtype, "| device", model.device)

## 5. Generation settings + batched helper
`GEN_TEMP = 1.0` matches Claude's EVAL 2 (temperature 1.0). Set it to `0.0` for greedy / deterministic decoding (the tuned model's single best output). `MAX_NEW_TOKENS` is generous; the tuned programs are compact — raise it only if you see truncated code.

In [ ]:
GEN_TEMP       = 1.0     # match Claude EVAL 2; use 0.0 for greedy/deterministic
MAX_NEW_TOKENS = 3072   # tuned programs are compact; raise if any get truncated
BATCH          = 8      # lower if you OOM on a small GPU

import torch
@torch.no_grad()
def generate_batch(pairs):
    """pairs: list of (system, user) -> list of completion strings (assistant turn only)."""
    outs = []
    for i in range(0, len(pairs), BATCH):
        chunk = pairs[i:i + BATCH]
        texts = [tok.apply_chat_template(
                    [{"role": "system", "content": s}, {"role": "user", "content": u}],
                    tokenize=False, add_generation_prompt=True)
                 for (s, u) in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=GEN_TEMP > 0, temperature=max(GEN_TEMP, 1e-5),
                             top_p=0.95, pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(new, skip_special_tokens=True))
        print(f"  generated {min(i + BATCH, len(pairs))}/{len(pairs)}", flush=True)
    return outs

## 6. EVAL 2 — generate on bare eval.jsonl prompts, score through the harness
Identical to Claude's EVAL 2: sizes 7/9/11/15, 25 prompts per size (n=100, drawn from `eval.jsonl` with the same seed), scored by `score_one` on the clean English palette with a real-dictionary check on every entry.

In [ ]:
import os, json, time
# these modules read ANTHROPIC_* via os.environ.get; set dummies so nothing complains
os.environ.setdefault("ANTHROPIC_BASE_URL", ""); os.environ.setdefault("ANTHROPIC_AUTH_TOKEN", "")
from pipeline.eval_opus_evalset import load_prompts
from pipeline.eval_opus_fleet import score_one, agg, table
from pipeline.eval_selfmodel import english_palette
from pipeline.eval_harness import extract_code

SIZES = [7, 9, 11, 15]; PER_SIZE = 25
prompts = load_prompts("data/sft/eval.jsonl", SIZES, PER_SIZE)   # (system, user, size), BARE
print(f"{len(prompts)} bare prompts")
print(f"  example -> system={prompts[0][0]!r}\n             user={prompts[0][1]!r}")
pal = english_palette(max(SIZES))

print("generating...", flush=True)
comps = generate_batch([(s, u) for (s, u, sz) in prompts])

print("scoring...", flush=True)
rows = []
for (s, u, sz), txt in zip(prompts, comps):
    code = extract_code(txt)
    if not code:
        rec = {"valid": 0, "fully": 0, "within": 0, "dict_frac": 0.0, "coverage": 0.0,
               "crossings": 0, "entries": 0, "filler": 0.0, "parsed": 0}
    else:
        rec = score_one(code, pal, sz, "vocabulary"); rec["parsed"] = 1
    rec["size"] = sz; rows.append(rec)

parse_rate = sum(r["parsed"] for r in rows) / len(rows)
print(f"\nparse rate (emitted a code block): {parse_rate*100:.0f}%")
ov = table(f"TUNED Qwen3-4B on eval.jsonl BARE prompts (n={len(rows)}, temp={GEN_TEMP})", rows, SIZES)

## 7. Save results + base-vs-tuned comparison

In [ ]:
import os, json, time
os.makedirs("runs/eval", exist_ok=True)
out = f"runs/eval/tuned_evalset_{int(time.time())}.json"
summary = {"model": "qwen3-4b-crossword-qlora-merged", "condition": "bare eval.jsonl prompts",
           "n": len(rows), "parse_rate": parse_rate, "gen_temp": GEN_TEMP, "overall": ov,
           "by_size": {s: agg([r for r in rows if r["size"] == s]) for s in SIZES}}
json.dump(summary, open(out, "w", encoding="utf-8"), indent=2)
print("wrote", out)
!mkdir -p /content/drive/MyDrive/slm_runs/eval && cp "$out" /content/drive/MyDrive/slm_runs/eval/ 2>/dev/null || true

print("\n===== EVAL 2 (bare eval.jsonl, harness-scored) — base vs tuned =====")
print(f"  Claude Opus 4.8 : valid  0%   fullyOK  0%   within  0%    [GAP_ANALYSIS EVAL 2, n=100]")
print(f"  Tuned Qwen3-4B  : valid {ov['valid']*100:3.0f}%   fullyOK {ov['fully']*100:3.0f}%   "
      f"within {ov['within']*100:3.0f}%   (n={len(rows)}, temp={GEN_TEMP})")

## 8. (Optional) dump the raw programs for inspection
Parallels Claude's saved as-is programs. Writes each emitted program to `runs/eval/tuned_progs/` so you can eyeball the actual generators the model produced.

In [ ]:
import os
os.makedirs("runs/eval/tuned_progs", exist_ok=True)
for k, ((s, u, sz), txt) in enumerate(zip(prompts, comps)):
    code = extract_code(txt) or txt
    open(f"runs/eval/tuned_progs/prog_{k:03d}_s{sz}.py", "w", encoding="utf-8").write(code)
print("saved", len(prompts), "programs under runs/eval/tuned_progs/")

## Next
If the tuned model clears a meaningful bar here (vs Claude's 0%), record the numbers in `GAP_ANALYSIS.md` as the tuned column of EVAL 2. To also run EVAL 1 (English clean-room) or the Extra Spanish eval, reuse `english_palette`/`spanish_palette` + `score_one` with the clean-room fleet prompt the same way.